# PEBBLE H1/H2 on Google Colab (A100)

**Claim file:** `experiments/pebble_reward_vs_rl/CLAIM.md`  
**Paper:** Lee et al., PEBBLE, ICML 2021 — Oracle teacher only (**not** B-Pref).

## Before you run
1. **Runtime → Change runtime type → GPU → A100**
2. If you already failed a `pip install` in this runtime: **Runtime → Disconnect and delete runtime**, then reconnect
3. Run cells **in order** (1 → 8). Do not skip Cell 4.

## How this notebook is laid out (Colab-specific)
| Path | Role |
|------|------|
| `/content/BPref` | Code clone on local SSD (fast) |
| Drive `.../LiraLab/pebble_exp` | Experiment outputs (survives disconnects) |
| `/content/pebble311` | Python **3.11** venv with pinned `gym==0.26.2` |

We **never** fall back to unpinned / latest `gym` — that changes env APIs.

## Status vocabulary
| Stage | Meaning |
|-------|---------|
| Smoke | wiring only — **not evidence** |
| Diagnostic | 5×4×100k — **tentative patterns only** |
| Paper-scale | 10×4×500k — claim gate |


In [ ]:
# @title Cell 1 — GPU check
import torch

assert torch.cuda.is_available(), (
    "No GPU attached. Runtime → Change runtime type → GPU → A100."
)

name = torch.cuda.get_device_name(0)
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {name} ({mem_gb:.1f} GB)")
print(f"torch: {torch.__version__}  cuda: {torch.version.cuda}")

if "A100" not in name.upper():
    print(
        "WARNING: not an A100. You can continue on T4/L4/V100 (slower), "
        "or reconnect until Colab assigns A100."
    )
else:
    print("A100 OK.")


In [ ]:
# @title Cell 2 — Paths (code on SSD, outputs on Drive)
from pathlib import Path
import os

# True = save diagnostics/checkpoints to Google Drive (recommended)
USE_DRIVE = True

CODE_ROOT = Path("/content/BPref")  # always local SSD for fast training I/O
REPO_URL = "https://github.com/thatrandomasiandev/BPref.git"
REPO_BRANCH = "main"

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    EXP_ROOT = Path("/content/drive/MyDrive/LiraLab/pebble_exp")
else:
    EXP_ROOT = Path("/content/pebble_exp")

EXP_ROOT.mkdir(parents=True, exist_ok=True)

# Persist paths for later cells / reconnects within the same VM
env_file = Path("/content/pebble_colab_env.py")
env_file.write_text(
    "from pathlib import Path\n"
    f"CODE_ROOT = Path({str(CODE_ROOT)!r})\n"
    f"EXP_ROOT = Path({str(EXP_ROOT)!r})\n"
    f"REPO_URL = {REPO_URL!r}\n"
    f"REPO_BRANCH = {REPO_BRANCH!r}\n"
)
print("CODE_ROOT =", CODE_ROOT)
print("EXP_ROOT  =", EXP_ROOT)
print("Wrote", env_file)


In [ ]:
# @title Cell 3 — Clone / update BPref fork onto /content
import os
import subprocess
from pathlib import Path

# Reload paths if you re-ran after a disconnect within the same VM
exec(open("/content/pebble_colab_env.py").read())

def sh(cmd: str, cwd=None):
    print(">>", cmd)
    subprocess.check_call(cmd, shell=True, cwd=cwd)


if (CODE_ROOT / ".git").exists():
    sh(
        f"git fetch origin {REPO_BRANCH} && "
        f"git checkout {REPO_BRANCH} && "
        f"git pull --ff-only origin {REPO_BRANCH}",
        cwd=str(CODE_ROOT),
    )
elif CODE_ROOT.exists() and any(CODE_ROOT.iterdir()):
    raise RuntimeError(
        f"{CODE_ROOT} exists but is not a git repo. "
        "Runtime → Disconnect and delete runtime, then re-run from Cell 1."
    )
else:
    sh(f"git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {CODE_ROOT}")

claim = CODE_ROOT / "experiments/pebble_reward_vs_rl/CLAIM.md"
assert claim.exists(), f"Missing {claim} — wrong repo/branch."

# Point Hydra outputs at Drive (or /content/pebble_exp) via symlink
exp_link = CODE_ROOT / "experiments/pebble_reward_vs_rl/exp"
if exp_link.is_symlink() or exp_link.exists():
    if exp_link.is_symlink():
        exp_link.unlink()
    elif exp_link.is_dir() and not any(exp_link.iterdir()):
        exp_link.rmdir()
    elif exp_link.exists() and not exp_link.is_symlink():
        # Keep any local outputs; move aside once
        bak = CODE_ROOT / "experiments/pebble_reward_vs_rl/exp_local_bak"
        if not bak.exists():
            exp_link.rename(bak)
            print("Moved existing local exp/ ->", bak)
        else:
            raise RuntimeError(f"Both {exp_link} and {bak} exist; clean up manually.")
exp_link.symlink_to(EXP_ROOT, target_is_directory=True)

os.chdir(CODE_ROOT)
print("cwd:", Path.cwd())
print("commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
print("exp symlink:", exp_link, "->", EXP_ROOT)


In [ ]:
# @title Cell 4 — Python 3.11 venv + pinned gym==0.26.2 (do not skip)
#
# Colab system Python is often 3.12/3.13. openai/gym==0.26.2 frequently fails
# to build there. Falling back to "latest gym" is forbidden for this study.
#
# Always use a dedicated 3.11 venv on Colab. Later cells read PEBBLE_PYTHON.
import os
import shutil
import subprocess
import sys
from pathlib import Path

exec(open("/content/pebble_colab_env.py").read())
os.chdir(CODE_ROOT)

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTHONPATH"] = f"{CODE_ROOT}:{CODE_ROOT / 'custom_dmc2gym'}"

VENV_DIR = Path("/content/pebble311")
PKGS = [
    "hydra-core",
    "omegaconf",
    "gym==0.26.2",
    "dm_control",
    "mujoco",
    "scikit-image",
    "tensorboard",
    "tqdm",
    "matplotlib",
]


def run(cmd, **kwargs):
    printable = " ".join(cmd) if isinstance(cmd, list) else cmd
    print(">>", printable, flush=True)
    subprocess.check_call(cmd, **kwargs)


def ensure_uv() -> str:
    uv = shutil.which("uv")
    if uv:
        return uv
    run("curl -LsSf https://astral.sh/uv/install.sh | sh", shell=True)
    for candidate in (
        Path.home() / ".local/bin/uv",
        Path.home() / ".cargo/bin/uv",
        Path("/root/.local/bin/uv"),
    ):
        if candidate.exists():
            os.environ["PATH"] = f"{candidate.parent}:{os.environ.get('PATH', '')}"
            return str(candidate)
    uv = shutil.which("uv")
    assert uv, "Failed to install uv"
    return uv


print("Colab system Python:", sys.version)
uv = ensure_uv()
run([uv, "python", "install", "3.11"])

if not (VENV_DIR / "bin/python").exists():
    run([uv, "venv", str(VENV_DIR), "--python", "3.11"])

PEBBLE_PYTHON = str(VENV_DIR / "bin/python")
print("PEBBLE_PYTHON =", PEBBLE_PYTHON)

# Install CUDA torch into the venv (Colab's system torch is not inherited).
try:
    run([
        uv, "pip", "install", "--python", PEBBLE_PYTHON,
        "torch", "--index-url", "https://download.pytorch.org/whl/cu124",
    ])
except subprocess.CalledProcessError:
    print("cu124 torch failed; retrying cu121 ...", flush=True)
    run([
        uv, "pip", "install", "--python", PEBBLE_PYTHON,
        "torch", "--index-url", "https://download.pytorch.org/whl/cu121",
    ])

run([uv, "pip", "install", "--python", PEBBLE_PYTHON, *PKGS])
run(
    [uv, "pip", "install", "--python", PEBBLE_PYTHON, "-e", "custom_dmc2gym"],
    cwd=str(CODE_ROOT),
)

# Persist paths for later cells (full rewrite — safe if this cell is re-run)
os.environ["PEBBLE_PYTHON"] = PEBBLE_PYTHON
Path("/content/pebble_colab_env.py").write_text(
    "from pathlib import Path\n"
    f"CODE_ROOT = Path({str(CODE_ROOT)!r})\n"
    f"EXP_ROOT = Path({str(EXP_ROOT)!r})\n"
    f"REPO_URL = {REPO_URL!r}\n"
    f"REPO_BRANCH = {REPO_BRANCH!r}\n"
    f"PEBBLE_PYTHON = {PEBBLE_PYTHON!r}\n"
)

# Hard checks — fail loud, never silently accept wrong gym
probe = (
    "import gym, torch, sys; "
    "print(gym.__version__); "
    "print(torch.__version__); "
    "print(torch.cuda.is_available()); "
    "print(sys.version.split()[0])"
)
out = subprocess.check_output([PEBBLE_PYTHON, "-c", probe], text=True).strip().splitlines()
gym_ver, torch_ver, cuda_ok, py_ver = out
print("verify gym   =", gym_ver)
print("verify torch =", torch_ver)
print("verify cuda  =", cuda_ok)
print("verify py    =", py_ver)

assert gym_ver.startswith("0.26"), f"Need gym 0.26.x, got {gym_ver}"
assert cuda_ok == "True", f"CUDA not visible inside venv (got {cuda_ok})"
assert py_ver.startswith("3.11"), f"Need Python 3.11.x in venv, got {py_ver}"
print("Dependency cell OK.")


In [ ]:
# @title Cell 5 — Smoke test (wiring only — NOT a finding)
import os
import subprocess
from pathlib import Path

exec(open("/content/pebble_colab_env.py").read())
os.chdir(CODE_ROOT)

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTHONPATH"] = f"{CODE_ROOT}:{CODE_ROOT / 'custom_dmc2gym'}"
os.environ["PEBBLE_PYTHON"] = PEBBLE_PYTHON

smoke_dir = EXP_ROOT / "walker_walk" / "full" / "seed0_colab_smoke"
smoke_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    PEBBLE_PYTHON,
    "experiments/pebble_reward_vs_rl/train_pebble_diagnostics.py",
    "device=cuda",
    "seed=0",
    "num_train_steps=6000",
    "env=walker_walk",
    "pebble_condition=full",
    "do_relabel=true",
    "num_seed_steps=1000",
    "num_unsup_steps=2000",
    "num_interact=2000",
    "max_feedback=80",
    "reward_batch=20",
    "reward_update=5",
    "eval_frequency=4000",
    "num_eval_episodes=1",
    "diag_holdout_pairs=64",
    "diag_onpolicy_pairs=32",
    "diag_probe_gradient_steps=20",
    "diag_probe_steps=[6000]",
    "agent.batch_size=256",
    f"hydra.run.dir={smoke_dir}",
]
print("SMOKE (wiring only):", " ".join(map(str, cmd)), flush=True)
subprocess.check_call(cmd, cwd=str(CODE_ROOT))

csv_path = smoke_dir / "diagnostics.csv"
assert csv_path.exists(), csv_path
print("Smoke OK ->", csv_path)
print("STATUS: wiring only. Not evidence.")


In [ ]:
# @title Cell 6 — Diagnostic suite (5 seeds x 4 conditions x 100k)
# CLAIM.md §6: tentative patterns only — not a paper-scale claim.
import os
import subprocess

exec(open("/content/pebble_colab_env.py").read())
os.chdir(CODE_ROOT)

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTHONPATH"] = f"{CODE_ROOT}:{CODE_ROOT / 'custom_dmc2gym'}"
os.environ["PEBBLE_PYTHON"] = PEBBLE_PYTHON
os.environ["DEVICE"] = "cuda"
os.environ["PARALLEL"] = "1"
os.environ["STEPS"] = "100000"
os.environ["SEEDS"] = "1 2 3 4 5"
os.environ["ENV"] = "walker_walk"

print("PEBBLE_PYTHON =", PEBBLE_PYTHON, flush=True)
print("EXP_ROOT =", EXP_ROOT, flush=True)
rc = subprocess.call(
    [PEBBLE_PYTHON, "experiments/pebble_reward_vs_rl/run_diagnostic_suite.py"],
    cwd=str(CODE_ROOT),
)
print("diagnostic suite exit:", rc)
if rc != 0:
    raise SystemExit(
        f"Diagnostic suite failed (rc={rc}). See "
        f"{CODE_ROOT}/experiments/pebble_reward_vs_rl/exp/_logs/"
    )


In [ ]:
# @title Cell 7 — Aggregate diagnostic results
import os
import subprocess
from pathlib import Path

exec(open("/content/pebble_colab_env.py").read())
os.chdir(CODE_ROOT)

subprocess.check_call(
    [
        PEBBLE_PYTHON,
        "experiments/pebble_reward_vs_rl/analyze_results.py",
        "--root",
        "experiments/pebble_reward_vs_rl/exp",
    ],
    cwd=str(CODE_ROOT),
)

summary = Path("experiments/pebble_reward_vs_rl/exp/_analysis/summary_by_condition.csv")
print(summary.read_text() if summary.exists() else "missing summary")
print()
print("STATUS: diagnostic aggregation only — not a justified H1/H2 conclusion.")
print("Outputs also under:", EXP_ROOT)


In [ ]:
# @title Cell 8 — Paper-scale suite (CLAIM.md claim gate)
# 10 seeds x 4 conditions x 500k. Many hours on A100. Resume-safe.
import os
import subprocess

RUN_PAPER_SCALE = False  # flip to True only after diagnostic looks sane

exec(open("/content/pebble_colab_env.py").read())
os.chdir(CODE_ROOT)

if not RUN_PAPER_SCALE:
    print("Skipping paper-scale (RUN_PAPER_SCALE=False). Set True and re-run this cell when ready.")
else:
    os.environ["MUJOCO_GL"] = "egl"
    os.environ["PYTHONUNBUFFERED"] = "1"
    os.environ["PYTHONPATH"] = f"{CODE_ROOT}:{CODE_ROOT / 'custom_dmc2gym'}"
    os.environ["PEBBLE_PYTHON"] = PEBBLE_PYTHON
    os.environ["DEVICE"] = "cuda"
    os.environ["PARALLEL"] = "1"
    os.environ["STEPS"] = "500000"
    os.environ["SEEDS"] = "1 2 3 4 5 6 7 8 9 10"
    os.environ["ENV"] = "walker_walk"

    print("Starting PAPER-SCALE with", PEBBLE_PYTHON, flush=True)
    rc = subprocess.call(
        [PEBBLE_PYTHON, "experiments/pebble_reward_vs_rl/run_paper_scale_suite.py"],
        cwd=str(CODE_ROOT),
    )
    print("paper-scale exit:", rc)
    if rc != 0:
        raise SystemExit(f"Paper-scale failed rc={rc}")
    subprocess.check_call(
        [
            PEBBLE_PYTHON,
            "experiments/pebble_reward_vs_rl/analyze_results.py",
            "--root",
            "experiments/pebble_reward_vs_rl/exp",
        ],
        cwd=str(CODE_ROOT),
    )


## After runs

- Outputs: Drive `MyDrive/LiraLab/pebble_exp/` (and via symlink under the repo `exp/`)
- Fill `RESULTS.md` from `_analysis/summary_by_condition.csv` using **CLAIM.md §7**
- Do **not** claim H1/H2 from smoke or incomplete seeds

### If Colab disconnects
1. Reconnect the same GPU type
2. Re-run Cells **1 → 4** (venv rebuild is mostly cached under `/content/pebble311`)
3. Re-run Cell 6 or 8 — suites **skip** completed `seed*` folders

### If Cell 4 fails
Do **not** `pip install gym` without a version pin. Delete the runtime and re-run Cell 4.
